# Reinforcement Learning for Stent Design Optimization

This notebook demonstrates using RL to optimize stent designs with the AutoStent framework.

## Goals
- Minimize stress (safety)
- Maximize compliance (flexibility)
- Minimize material volume (efficiency)

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# AutoStent framework
from autostent.geometry import SplineStentGeometry, StentGeometryParameters
from autostent.rl import StentDesignEnv
from autostent.evaluation import compute_biomechanical_metrics

# RL libraries
try:
    from stable_baselines3 import PPO
    from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
    SB3_AVAILABLE = True
except ImportError:
    print('stable-baselines3 not installed. Install with: pip install stable-baselines3')
    SB3_AVAILABLE = False

print('Imports successful')

In [ ]:
# 4C Solver Configuration
# -----------------------
# Using Docker-based execution (ghcr.io/4c-multiphysics/4c:main)

# Path to local 4C executable (used when use_docker=False)
fourc_path = 'fourc'

# Set to True to use the Docker container (recommended)
use_docker = True

# Set to True to use fake results (for testing without Docker/4C)
use_mock_simulation = False

print(f"fourc_path: {fourc_path}")
print(f"Docker Mode: {use_docker}")
print(f"Mock Simulation: {use_mock_simulation}")

## 2. Understanding the RL Environment

In [ ]:
# Create environment
env = StentDesignEnv(
    fourc_executable=fourc_path,  # Uses the path defined above
    mock_simulation=use_mock_simulation,
    use_docker=use_docker,
    max_episode_steps=50,
)

print(f'Observation space: {env.observation_space}')
print(f'Action space: {env.action_space}')

# Test environment
obs, info = env.reset()
print(f'\nInitial observation shape: {obs.shape}')
print(f'Initial parameters: {info["parameters"]}')

# Take a random action
action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)
print(f'\nAfter one step:')
print(f'  Reward: {reward:.4f}')
print(f'  Terminated: {terminated}')

## 3. Baseline: Random Search

In [ ]:
def random_search_baseline(env, num_episodes=200):
    rewards = []
    best_reward = -np.inf
    best_params = None
    
    for episode in range(num_episodes):
        obs, info = env.reset()
        episode_reward = 0
        
        for step in range(env.max_episode_steps):
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            
            if reward > best_reward:
                best_reward = reward
                best_params = info['parameters'].copy()
            
            if terminated or truncated:
                break
        
        rewards.append(episode_reward)
        if (episode + 1) % 5 == 0:
            print(f'Episode {episode + 1}/{num_episodes}, Avg Reward: {np.mean(rewards[-5:]):.4f}')
    
    return rewards, best_reward, best_params

print('Running random search baseline...')
baseline_rewards, baseline_best, baseline_params = random_search_baseline(env, num_episodes=20)
print(f'\nBaseline Results:')
print(f'  Average reward: {np.mean(baseline_rewards):.4f} +/- {np.std(baseline_rewards):.4f}')
print(f'  Best reward: {baseline_best:.4f}')

## 4. Training RL Agent with PPO

In [ ]:
if not SB3_AVAILABLE:
    print('Skipping RL training - stable-baselines3 not available')
else:
    # Create fresh environments
    train_env = StentDesignEnv(fourc_executable=fourc_path, mock_simulation=use_mock_simulation,
    use_docker=use_docker,)  # Uses the path defined above
    eval_env = StentDesignEnv(fourc_executable=fourc_path, mock_simulation=use_mock_simulation,
    use_docker=use_docker,)  # Uses the path defined above
    
    # Initialize PPO agent
    model = PPO(
        'MlpPolicy',
        train_env,
        verbose=1,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        tensorboard_log=None,
        device='cpu',  # Use Apple GPU
    )
    
    print('PPO agent initialized')
    
    # Setup callbacks
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path='./models/stent_ppo/best/',
        log_path='./logs/stent_ppo/',
        eval_freq=5000,
        deterministic=True,
        render=False,
    )
    
    checkpoint_callback = CheckpointCallback(
        save_freq=10000,
        save_path='./models/stent_ppo/checkpoints/',
        name_prefix='stent_ppo',
    )
    
    print('\nStarting training (this will take several minutes)...')
    
    # Train
    model.learn(
        total_timesteps=10000,
        callback=[eval_callback, checkpoint_callback],
        progress_bar=True,
    )
    
    print('\nTraining completed!')
    model.save('./models/stent_ppo/final_model')
    print('Model saved')


## 5. Evaluation

In [ ]:
if not SB3_AVAILABLE:
    rl_rewards = baseline_rewards
    rl_best = baseline_best
    rl_params = baseline_params
else:
    # Load model
    try:
        model = PPO.load('./models/stent_ppo/best/best_model', env=eval_env)
        print('Loaded best model')
    except:
        try:
            model = PPO.load('./models/stent_ppo/final_model', env=eval_env)
            print('Loaded final model')
        except:
            print('No model found, using random policy')
            model = None
    
    # Evaluate
    rl_rewards = []
    rl_best = -np.inf
    rl_params = None
    
    for episode in range(20):
        obs, info = eval_env.reset()
        episode_reward = 0
        
        for step in range(eval_env.max_episode_steps):
            if model:
                action, _ = model.predict(obs, deterministic=True)
            else:
                action = eval_env.action_space.sample()
            
            obs, reward, terminated, truncated, info = eval_env.step(action)
            episode_reward += reward
            
            if reward > rl_best:
                rl_best = reward
                rl_params = info['parameters'].copy()
            
            if terminated or truncated:
                break
        
        rl_rewards.append(episode_reward)
    
    print(f'\nRL Agent Results:')
    print(f'  Average reward: {np.mean(rl_rewards):.4f} +/- {np.std(rl_rewards):.4f}')
    print(f'  Best reward: {rl_best:.4f}')
    
    # Compare
    improvement = ((np.mean(rl_rewards) - np.mean(baseline_rewards)) / abs(np.mean(baseline_rewards))) * 100
    print(f'\nImprovement: {improvement:+.2f}%')

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Success Rate Comparison
ax1 = axes[0, 0]
baseline_success = np.mean(np.array(baseline_rewards) > -100) * 100
rl_success = np.mean(np.array(rl_rewards) > -100) * 100
bars = ax1.bar(['Random Search', 'RL Agent'], [baseline_success, rl_success], color=['blue', 'orange'], alpha=0.7)
ax1.set_ylabel('Success Rate (%)')
ax1.set_title('Design Validation Success Rate')
ax1.set_ylim(0, 100)
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom')
ax1.grid(True, alpha=0.3)

# 2. Filtered Reward Distribution (Valid Designs Only)
ax2 = axes[0, 1]
valid_baseline = [r for r in baseline_rewards if r > -99]
valid_rl = [r for r in rl_rewards if r > -99]

if valid_baseline or valid_rl:
    if valid_baseline:
        ax2.hist(valid_baseline, bins=10, alpha=0.6, label=f'Random (n={len(valid_baseline)})', color='blue')
    if valid_rl:
        ax2.hist(valid_rl, bins=10, alpha=0.6, label=f'RL (n={len(valid_rl)})', color='orange')
    ax2.set_xlabel('Reward (Valid Designs Only)')
    ax2.set_ylabel('Count')
    ax2.set_title('Reward Distribution (Excluding Failures)')
    ax2.legend()
else:
    ax2.text(0.5, 0.5, 'No Valid Designs Found\n(All rewards <= -100)', 
             ha='center', va='center', fontsize=12)
    ax2.set_title('Reward Distribution (Excluding Failures)')
ax2.grid(True, alpha=0.3)

# 3. Overall Reward comparison (Boxplot)
ax3 = axes[1, 0]
ax3.boxplot([baseline_rewards, rl_rewards], tick_labels=['Random Search', 'RL Agent'])
ax3.set_ylabel('Episode Reward')
ax3.set_title('Overall Performance Comparison (Including Failures)')
ax3.grid(True, alpha=0.3)

# 4. Parameter Space Exploration
ax4 = axes[1, 1]
ax4.axis('off')
ax4.text(0.5, 0.5, 'Parameter Analysis Placeholder\n(Requires tracking episode params)', 
         ha='center', va='center', fontsize=12)
ax4.set_title('Design Space Exploration')

plt.tight_layout()
plt.savefig('stent_rl_results_improved.png', dpi=150)
print('Results saved to stent_rl_results_improved.png')
plt.show()

## Summary

This notebook demonstrated:
1. RL environment setup for stent design
2. Baseline evaluation with random search
3. PPO training for design optimization
4. Performance comparison and visualization